# skillspan-ner: experiment sweep, round 3 (2 GPUs in parallel)

So far (test set, Viterbi decoding, mean +/- std over seeds):
- BERT 0.558 +/- 0.004 (2 seeds), JobBERT 0.598 +/- 0.007 (3), **DeBERTa-v3 0.625 +/- 0.005 (3)**
- SkillSpan paper's best combined: 59.73 (JobBERT, single-task, CRF, 5 seeds)
- Gap: our SKILL F1 (DeBERTa 0.540) is below the paper's 56.11; KNOWLEDGE (0.697) is above its 63.88.

Round 3 (DeBERTa-v3):

| runs | question |
|---|---|
| `deberta-crf-s42/43/44` | does a learned CRF (like the paper) improve long SKILL spans? |
| `deberta-e10-s42/43` | does longer training (10 epochs vs 5) help? |

Also checks on the **validation** set that the decoding choice (argmax vs Viterbi vs CRF) holds there, not just on test.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.

In [ ]:
!git clone -q https://github.com/MSR327/skillspan-ner.git
%cd skillspan-ner
!pip install -q -U "transformers>=5.0" "accelerate>=1.1" datasets sentencepiece
!pip install -q -e ".[train]"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## Pre-download models and data (so the parallel workers don't race on downloads)

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

for m in ["jjzha/jobbert-base-cased", "jjzha/jobspanbert-base-cased", "microsoft/deberta-v3-base"]:
    snapshot_download(m, allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.bin"])
    print("cached", m)
print({k: len(v) for k, v in load_dataset("jjzha/skillspan").items()})

## Run the sweep: one worker per GPU

In [ ]:
import os
import subprocess
import threading
import time
from pathlib import Path

COMMON = ["batch_size=32"]
EXPERIMENTS = {
    # Round 3. DeBERTa-v3 is the best so far (round 2: 0.625 +/- 0.005 vit_f1, 3 seeds).
    0: [  # GPU 0: learned CRF, 5 epochs
        ("deberta-crf-s42", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "crf=true", "seed=42"]),
        ("deberta-crf-s43", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "crf=true", "seed=43"]),
        ("deberta-crf-s44", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "crf=true", "seed=44"]),
    ],
    1: [  # GPU 1: no CRF, 10 epochs
        ("deberta-e10-s42", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "epochs=10", "seed=42"]),
        ("deberta-e10-s43", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "epochs=10", "seed=43"]),
    ],
}
Path("logs").mkdir(exist_ok=True)
status = {}


def worker(gpu, jobs):
    for name, sets in jobs:
        t0 = time.time()
        cmd = ["python", "-m", "skillex.train", "--config", "configs/jobbert.yaml", "--set",
               f"run_name={name}", f"output_dir=outputs/{name}", *COMMON, *sets]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        with open(f"logs/{name}.log", "w") as f:
            # check=False: a failed run must not stop the others
            proc = subprocess.run(cmd, check=False, env=env, stdout=f, stderr=subprocess.STDOUT)
        rc = proc.returncode
        status[name] = "OK" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[gpu{gpu}] {name}: {status[name]} in {(time.time() - t0) / 60:.1f} min", flush=True)


threads = [threading.Thread(target=worker, args=(g, jobs)) for g, jobs in EXPERIMENTS.items()]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(status)

## Failures (last 25 log lines of any failed run)

In [ ]:
for name, s in status.items():
    if s != "OK":
        print(f"===== {name} =====")
        print("\n".join(Path(f"logs/{name}.log").read_text().splitlines()[-25:]))

## Results (test set, exact span-match)

In [ ]:
import json

import pandas as pd

rows = []
for f in sorted(Path("metrics").glob("*.json")):
    m = json.loads(f.read_text())
    c = m["config"]
    rows.append({
        "run": f.stem, "model": c["base_model"].split("/")[-1], "seed": c["seed"],
        "lr": f'{c["learning_rate"]:.0e}', "epochs": c["epochs"],
        "f1": m["test_f1"], "vit_f1": m["test_vit_f1"], "crf_f1": m.get("test_crf_f1"),
        "precision": m["test_precision"], "vit_precision": m["test_vit_precision"],
        "recall": m["test_recall"],
        "f1_skill": m["test_f1_skill"], "vit_f1_skill": m["test_vit_f1_skill"], "crf_f1_skill": m.get("test_crf_f1_skill"),
        "f1_knowledge": m["test_f1_knowledge"], "vit_f1_knowledge": m["test_vit_f1_knowledge"], "crf_f1_knowledge": m.get("test_crf_f1_knowledge"),
    })
df = pd.DataFrame(rows).set_index("run").sort_values("vit_f1", ascending=False)
pd.set_option("display.width", 250)
print(df.round(4).to_string())

## Mean +/- std per model family (across seeds)

In [ ]:
df["family"] = df.index.str.replace(r"-s\d+$", "", regex=True)
cols = ["f1", "vit_f1", "crf_f1", "precision", "vit_precision", "f1_skill", "vit_f1_skill", "crf_f1_skill", "f1_knowledge", "vit_f1_knowledge", "crf_f1_knowledge"]
summary = df.groupby("family")[cols].agg(["mean", "std", "count"])
print(summary.round(4).to_string())

## Validation check: does the decoding choice hold on the validation set?

Best epoch per run (the checkpoint that was kept). If Viterbi/CRF beat argmax here too, the decision isn't based on the test set.

In [ ]:
val_rows = []
for f in sorted(Path("metrics").glob("*.json")):
    m = json.loads(f.read_text())
    hist = m.get("val_history") or []
    if not hist:
        continue
    key = "eval_crf_f1" if "eval_crf_f1" in hist[0] else "eval_f1"
    best = max(hist, key=lambda e: e[key])
    val_rows.append({"run": f.stem, "best_epoch": best["epoch"], "val_f1": best["eval_f1"],
                     "val_vit_f1": best["eval_vit_f1"], "val_crf_f1": best.get("eval_crf_f1")})
vdf = pd.DataFrame(val_rows).set_index("run")
vdf["vit_gain"] = vdf["val_vit_f1"] - vdf["val_f1"]
print(vdf.round(4).to_string())